### CRUD Endpoints Overview

CRUD stands for:

| Operation | HTTP Method  | Description               |
|-----------|-------------|---------------------------|
| Create    | POST        | Add a new resource        |
| Read      | GET         | Retrieve existing resource(s) |
| Update    | PUT / PATCH | Modify existing resource  |
| Delete    | DELETE      | Remove resource           |

## Idempotency & Safe Methods

- **Safe methods:** Methods that do not modify data.

   - **Example:** GET, HEAD

   - You can call them repeatedly without changing the state.

- **Idempotent methods:** Methods that can be called multiple times and have the same effect as calling once.

  - **Example:** PUT, DELETE

POST is not idempotent (each call creates a new resource).

### Naming & Structure of CRUD Routes

**Principle:** Use nouns for resources, plural form is standard, avoid verbs.

| Operation       | Endpoint Pattern   | Example                    |
|-----------------|-----------------|----------------------------|
| Create          | POST /items/      | Add a new item             |
| Read all        | GET /items/       | Retrieve all items         |
| Read one        | GET /items/{id}   | Retrieve a specific item   |
| Update          | PUT /items/{id}   | Replace a specific item    |
| Partial Update  | PATCH /items/{id} | Update fields of an item   |
| Delete          | DELETE /items/{id}| Remove a specific item     |

### Separating "Read One" vs "Read List"
- **Read List (Collection):**

   - GET /items/ → returns all items (or filtered, paginated)

- **Read One (Single Resource):**

    - GET /items/{id} → returns only one item.

## Example in FastAPI

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List

app = FastAPI()

# Model
class Item(BaseModel):
    id: int
    name: str
    price: float

# In-memory storage
items_db = {}

# CREATE
@app.post("/items/", response_model=Item)
def create_item(item: Item):
    if item.id in items_db:
        raise HTTPException(status_code=400, detail="Item already exists")
    items_db[item.id] = item
    return item

# READ LIST
@app.get("/items/", response_model=List[Item])
def read_items():
    return list(items_db.values())

# READ ONE
@app.get("/items/{item_id}", response_model=Item)
def read_item(item_id: int):
    if item_id not in items_db:
        raise HTTPException(status_code=404, detail="Item not found")
    return items_db[item_id]

# UPDATE
@app.put("/items/{item_id}", response_model=Item)
def update_item(item_id: int, item: Item):
    if item_id not in items_db:
        raise HTTPException(status_code=404, detail="Item not found")
    items_db[item_id] = item
    return item

# DELETE
@app.delete("/items/{item_id}")
def delete_item(item_id: int):
    if item_id not in items_db:
        raise HTTPException(status_code=404, detail="Item not found")
    del items_db[item_id]
    return {"detail": "Deleted successfully"}